In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

# ML Functionalities

In [10]:
def train_model(typ, model, lossf, n_epochs, opt, train_dataloader, val_dataloader=None, scheduler=None, earlyStop=None, verbose=10):
    train_lossLog = []
    val_lossLog = []
    best_loss, best_epoch = 1000, 0
    for epoch in range(1, n_epochs+1):
        train_lossSum = 0
        for batch in train_dataloader:
            if typ.lower() == "gcn":
                x, y = batch.x.float(), batch.y.float()
                y_predict = model(batch.x, batch.edge_index, batch.batch)
                y = batch.y.view(batch.num_graphs, -1)
            else:
                x, y = batch[:][0].float(), batch[:][1].float()
                y_predict = model(x)
            loss = lossf(y_predict, y)
            train_lossSum += loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            train_lossLog.append(loss.item())
        train_lossAvg = train_lossSum/len(train_dataloader)
        
        if val_dataloader:
            with torch.no_grad():
                val_lossSum = 0
                for batch in val_dataloader:
                    if typ.lower() == "gcn":
                        x, y = batch.x.float(), batch.y.float()
                        y_predict = model(batch.x, batch.edge_index, batch.batch)
                        y = batch.y.view(batch.num_graphs, -1)
                    else:
                        x, y = batch[:][0].float(), batch[:][1].float()
                        y_predict = model(x)
                    loss = lossf(y_predict, y)

                    val_lossLog.append(loss.item())
                    val_lossSum += loss

                val_lossAvg = val_lossSum/len(val_dataloader)
            if scheduler:
                scheduler.step(val_lossAvg)
            if earlyStop:
                earlyStop(val_lossAvg)
                if earlyStop.early_stop:
                    break
        else:
            if scheduler:
                scheduler.step(train_lossAvg)
            if earlyStop:
                earlyStop(train_lossAvg)
                if earlyStop.early_stop:
                    break
        
        if torch.abs(loss) < best_loss:
            best_loss = torch.abs(loss).item()
            best_epoch = epoch
        
        if verbose:
            if epoch == 1 or epoch % int(verbose) == 0:
                print("Epoch:", epoch, "- Loss:", loss.item())
            
    print(f"Best Epoch: {best_epoch}, with loss {best_loss}")
    # torch.save(model.state_dict(), "new_best_deep_ritz1.mdl")
    return model, epoch, train_lossLog, val_lossLog

def predict_model(typ, model, test_dataloader):
    test_outputs = []
    with torch.no_grad():
        for batch in test_dataloader:
            if typ.lower() == "gcn":
                x, y = batch.x.float(), batch.y.float()
                y_predict = model(batch.x, batch.edge_index, batch.batch)
                y = batch.y.view(batch.num_graphs, -1)
            else:
                x, y = batch[:][0].float(), batch[:][1].float()
                y_predict = model(x)
            test_outputs.append(y_predict.detach().cpu().numpy())
    test_outputs = np.concatenate(test_outputs)
    return test_outputs

In [4]:
def weights_init(m, dist="xavier"):
    if isinstance(m, nn.Linear):
        #nn.init.xavier_normal_(m.weight)
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.constant_(m.bias, 0.0)

In [5]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0001, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        
        if self.counter >= self.patience:
            if self.verbose:
                print(f"Early stopping triggered after {self.patience} epochs without improvement.")
            self.early_stop = True  # Set flag for stopping

## Custom Loss Functions

In [6]:
def QuantileLoss():
    pass

def custom_loss(target, output):   ### TODO: Quantile loss
    return torch.mean((output - target)**2)

def physics_loss(target, output):
    return torch.abs(torch.sum(target) - torch.sum(output))

## GNN Functions

In [7]:
def visualize_graph(loader, colors=None, layout="kk"):
    for batch in loader:
        dat = batch.get_example(0)
        break
    G = to_networkx(dat, to_undirected=True)
    
    plt.figure(figsize=(7,7))
    if layout.lower() == "kk":
        pos = nx.kamada_kawai_layout(G)
    elif layout.lower() == "spec":
        pos = nx.spectral_layout(G)
    elif layout.lower() == "spring":
        pos = nx.spring_layout(G, seed=1)
    elif layout.lower() == "planar":
        pos = nx.planar_layout(G)
    elif layout.lower() == "rand":
        pos = nx.random_layout(G)
    
    if colors is None:
        colors = dat.x[:, 0].detach().numpy()
    
    nx.draw_networkx_nodes(G, pos, node_size=20, node_color=colors, cmap='viridis')
    nx.draw_networkx_edges(G, pos, alpha=0.5, width=0.5)
    
    plt.show()

# Result Visualization

# Error Visualization

In [ ]:
def plot_loss(epoch, train, val):
    fig = plt.figure(figsize=(10, 5))
    plt.plot(np.linspace(1, epoch, len(train)), train)
    plt.plot(np.linspace(1, epoch, len(val)), val)
    plt.xlabel("Training Epoch")
    plt.ylabel("Mean Squared Error (MSE)")
    plt.yscale("log")
    title = plt.title("Training and Validation Loss Vs Epoch")
    legend = plt.legend(["Training", "Validation"])
    plt.grid()